In [1]:
"""
Board Detection Training Notebook

This notebook trains a YOLOv8 model to detect board games in images.
The pipeline includes dataset preparation, train/val splitting, model training,
validation, and model export.

Author: NCSU - Fall 2025 Board Game Project
"""

# ============================================================================
# IMPORTS
# ============================================================================

import os
import shutil
import random
from pathlib import Path
from ultralytics import YOLO  # YOLOv8 implementation
import yaml  # For creating configuration files
import torch  # PyTorch deep learning framework
from torch.serialization import add_safe_globals
from ultralytics.nn.tasks import DetectionModel

In [2]:
# ============================================================================
# STEP 1: SETUP DATASET PATHS
# ============================================================================

# Define base directory containing the dataset
base_dir = Path("/mnt/e/NCSU/Fall_2025/Board Game/datasets/bounding_box_dataset")

print(base_dir)

# Paths to augmented images and labels (created by data_augmentation.py)
images_dir = base_dir / "images_aug"
labels_dir = base_dir / "labels_aug"

# Output directory structure for YOLO training
output_dir = base_dir / "yolo_dataset"
train_images = output_dir / "train/images"
train_labels = output_dir / "train/labels"
val_images   = output_dir / "val/images"
val_labels   = output_dir / "val/labels"

# Create output directories (uncomment to run)
# This will create the train/val directory structure
# for d in [train_images, train_labels, val_images, val_labels]:
#     d.mkdir(parents=True, exist_ok=True)

/mnt/e/NCSU/Fall_2025/Board Game/datasets/bounding_box_dataset


In [ ]:
# ============================================================================
# STEP 2: TRAIN/VALIDATION SPLIT
# ============================================================================

# Collect all image files from augmented dataset
all_images = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

# Set random seed for reproducible splits
random.seed(42)
random.shuffle(all_images)

# Split dataset: 80% training, 20% validation
split_idx = int(0.8 * len(all_images))
train_files = all_images[:split_idx]
val_files   = all_images[split_idx:]

# Function to copy image and label files to train/val directories
# def copy_files(file_list, dest_img_dir, dest_lbl_dir):
#     """Copy images and their corresponding label files to destination"""
#     for img_path in file_list:
#         # Find corresponding label file
#         lbl_path = labels_dir / (img_path.stem + ".txt")
#         if lbl_path.exists():
#             # Copy both image and label
#             shutil.copy(img_path, dest_img_dir / img_path.name)
#             shutil.copy(lbl_path, dest_lbl_dir / lbl_path.name)

# Copy files to their respective directories (uncomment to run)
# copy_files(train_files, train_images, train_labels)
# copy_files(val_files, val_images, val_labels)

# print(f"Copied {len(train_files)} train and {len(val_files)} val images.")

In [ ]:
# ============================================================================
# STEP 3: CREATE YOLO CONFIGURATION FILE (data.yaml)
# ============================================================================

# YOLO requires a data.yaml file that specifies:
# - Paths to training and validation datasets
# - Number of classes (nc)
# - Class names

# data_yaml = {
#     "train": str(train_images.parent.resolve()),  # Path to train directory
#     "val": str(val_images.parent.resolve()),      # Path to val directory
#     "nc": 1,                                      # Number of classes (1 = board)
#     "names": ["board"]                            # Class name(s)
# }

# Write configuration to data.yaml file (uncomment to run)
# with open(output_dir / "data.yaml", "w") as f:
#     yaml.dump(data_yaml, f, default_flow_style=False)

# print(f"data.yaml created at {output_dir / 'data.yaml'}")

data.yaml created at Dataset\yolo_dataset\data.yaml


In [3]:
# ============================================================================
# STEP 4: CHECK FOR GPU AVAILABILITY
# ============================================================================

# Check if CUDA-compatible GPU is available
# Training on GPU is significantly faster than CPU
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"  # Fall back to CPU
    print("⚠️ CUDA not available, falling back to CPU")
    print("Note: Training on CPU will be much slower. Consider using Google Colab with GPU.")

✅ Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
# ============================================================================
# STEP 5: TRAIN YOLO MODEL
# ============================================================================

# Load pre-trained YOLOv8 nano model
# yolov8n.pt is the smallest/fastest model
# For better accuracy, try: yolov8s.pt, yolov8m.pt, or yolov8l.pt
model = YOLO("yolov8n.pt")

# Train the model with specified parameters
results = model.train(
    data=str(output_dir / "data.yaml"),  # Path to data configuration file
    epochs=50,                           # Number of training epochs (increase for better results)
    imgsz=640,                          # Input image size (smaller = faster, larger = better accuracy)
    batch=4,                            # Batch size (reduce if out of memory)
    device=device,                       # Device: "cpu" or 0 for GPU
    workers=0,                          # Number of data loader workers (0 = disable multiprocessing)
    cache=True,                         # Cache images in RAM for faster training
    patience=10,                         # Early stopping: stop if no improvement for 5 epochs
    name="board-detector",              # Name for this training run
    exist_ok=True,                      # Allow overwriting existing run
    verbose=True                        # Print training progress
)

# Training outputs:
# - Model checkpoints saved to: runs/detect/board-detector/weights/
# - Training logs and visualizations in: runs/detect/board-detector/
# - Best model: best.pt, Last model: last.pt

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.209 🚀 Python-3.10.18 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/e/NCSU/Fall_2025/Board Game/datasets/bounding_box_dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

In [5]:
# ============================================================================
# STEP 6: VALIDATE MODEL PERFORMANCE
# ============================================================================

# Run validation on the validation set
# This calculates metrics like precision, recall, mAP (mean Average Precision)
metrics = model.val()

# Print validation metrics
print("\n=== Validation Metrics ===")
print(metrics)
print("\nKey metrics to check:")
print("- mAP50: Mean Average Precision at IoU threshold 0.5")
print("- mAP50-95: Mean Average Precision averaged over IoU 0.5-0.95")
print("- Precision: Accuracy of positive predictions")
print("- Recall: Fraction of actual boards detected")

Ultralytics 8.3.209 🚀 Python-3.10.18 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 3.7±0.3 ms, read: 119.2±8.6 MB/s, size: 5174.3 KB)
val: Scanning /mnt/e/NCSU/Fall_2025/Board Game/datasets/bounding_box_dataset/yolo_dataset/val/labels.cache... 104 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 104/104 131.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 0.8it/s 9.2s0.6ss
                   all        104        104      0.999          1      0.995      0.934
Speed: 1.2ms preprocess, 10.5ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /mnt/e/NCSU/Fall_2025/Board Game/superpoint/src/runs/detect/val

=== Validation Metrics ===
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric o

In [ ]:
# ============================================================================
# STEP 7: TEST INFERENCE ON SAMPLE IMAGE
# ============================================================================

# Run inference on a random validation image to test the model
test_img = random.choice(val_files)
pred = model(str(test_img), save=True)

print(f"Inference saved for {test_img.name}")
print(f"Results saved to: runs/detect/predict/")
print("\nTo view the result, check the generated image with bounding boxes.")


image 1/1 e:\NCSU\Fall 2025\Board Game\Dataset\images_aug\7045530a-3-2050_aug2.jpg: 320x416 1 board, 34.0ms
Speed: 2.0ms preprocess, 34.0ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 416)
Results saved to runs\detect\board-detector


Inference saved for 7045530a-3-2050_aug2.jpg


In [6]:
# ============================================================================
# STEP 8: SAVE TRAINED MODEL
# ============================================================================

# Create directory for saving models
model_save_dir = Path("saved_models")
model_save_dir.mkdir(exist_ok=True)

# Copy the best weights from training to saved_models directory
# The best.pt file contains the model weights from the epoch with best validation performance
model_path = model_save_dir / "board_detector_v1.pt"
shutil.copy("runs/detect/board-detector/weights/best.pt", model_path)
print(f"✅ Model saved to {model_path}")

# ============================================================================
# LOADING AND USING THE SAVED MODEL
# ============================================================================

# # Load the saved model for inference
# loaded_model = YOLO(model_path)

# def detect_board(image_path):
#     """
#     Detect boards in an image using the trained model.
    
#     Args:
#         image_path (str): Path to the image file
        
#     Returns:
#         Detection results containing bounding boxes, confidence scores, etc.
#     """
#     results = loaded_model(image_path)
#     return results[0]  # Return first (and only) detection result

# # Test the loaded model on a validation image
# test_img = random.choice(val_files)
# result = detect_board(str(test_img))
# print(f"\n=== Detection Test ===")
# print(f"Test image: {test_img.name}")
# print(f"Detected {len(result.boxes)} boards in test image")

# # Access detection details
# if len(result.boxes) > 0:
#     for i, box in enumerate(result.boxes):
#         print(f"\nBoard {i+1}:")
#         print(f"  - Confidence: {box.conf[0]:.3f}")
#         print(f"  - Bounding box: {box.xyxy[0].tolist()}")  # [x1, y1, x2, y2]

✅ Model saved to saved_models/board_detector_v1.pt
